# NB2 — Plutchik Labeling: GoEmotions Part 2 + SemEval EI-reg
**Person 2 runs this notebook.** Labels the second half of GoEmotions, then loads SemEval EI-reg continuous intensity scores (already in 0-1 range) for joy/sadness/fear/anger.

- GPU: T4 x1 (Settings → Accelerator)
- Expected runtime: ~2 hours
- Outputs:
  - `/kaggle/working/plutchik_labeled_p2.csv` — LLM-labeled GoEmotions part 2
  - `/kaggle/working/semeval_continuous.csv` — SemEval EI-reg continuous labels

> After completion: share notebook output as Kaggle dataset named `plutchik-labels-p2`


## 0. Install

In [ ]:
%%capture
!pip install -U requests -q
!pip install datasets==2.19.0 -q
!pip install transformers accelerate bitsandbytes sentencepiece -q
print('done')

## 1. Config

In [ ]:
import os, json, re, torch
import pandas as pd
import numpy as np
from datasets import load_dataset

PART_START   = 22000      # start index in GoEmotions train split
PART_END     = None       # to the end
SAVE_PATH_P2 = '/kaggle/working/plutchik_labeled_p2.csv'
SAVE_PATH_SE = '/kaggle/working/semeval_continuous.csv'
CKPT_PATH    = '/kaggle/working/ckpt_p2.csv'
MODEL_ID     = 'Qwen/Qwen2.5-3B-Instruct'
BATCH_SIZE   = 16     # T4 has headroom with 4-bit (~2GB model, 16GB VRAM)
MAX_NEW_TOK  = 120    # needs 120 tokens to complete all 9 JSON keys
EMOTIONS     = ['joy','trust','fear','surprise','sadness','disgust','anger','anticipation']
print('Config OK  |  GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Load Model (4-bit NF4 on T4 — ~2 GB VRAM)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
print('Loading model (4-bit NF4)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()
print(f'Model loaded | device: {next(model.parameters()).device}')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 3. Prompt + Parser

In [ ]:
SYSTEM_PROMPT = """You are an expert emotion analyst. Given a sentence, output ONLY a valid JSON object with these exact 9 keys:
joy, trust, fear, surprise, sadness, disgust, anger, anticipation, confidence

Rules:
- All values must be floats between 0.0 and 1.0
- Scores reflect RELATIVE intensity across all 8 emotions
- confidence = how clearly the text expresses any emotion
- Do NOT output any explanation — JSON only"""


def build_prompt(sentence: str) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nSentence: {sentence.strip()}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def parse_output(raw: str) -> dict | None:
    """Robust parser — handles single quotes, wrong case, extra text around JSON."""
    raw = raw.strip()
    m = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
    if not m:
        return None
    try:
        blob = m.group()
        blob = blob.replace("'", '"')
        blob = re.sub(r'(?<!["\w])(\w+)\s*:', r'"\1":', blob)
        blob = re.sub(r'""(\w+)""', r'"\1"', blob)
        obj = json.loads(blob)
        obj_lower = {k.lower().strip(): v for k, v in obj.items()}
        required = set(EMOTIONS + ['confidence'])
        if not required.issubset(obj_lower.keys()):
            return None
        return {k: max(0.0, min(1.0, float(obj_lower[k]))) for k in required}
    except Exception:
        return None


print('Prompt + parser ready')

## 4. Batched Inference

In [ ]:
from tqdm.auto import tqdm


def label_batch(sentences):
    prompts = [build_prompt(s) for s in sentences]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True,
                        truncation=True, max_length=192).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOK,
            do_sample=False,              # greedy — faster + more consistent JSON
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    plen = inputs['input_ids'].shape[1]
    return [parse_output(tokenizer.decode(seq[plen:], skip_special_tokens=True)) for seq in out]


def label_dataset(sentences, batch_size=BATCH_SIZE):
    rows, n_fail = [], 0
    for i in tqdm(range(0, len(sentences), batch_size), desc='Labeling'):
        batch = sentences[i:i+batch_size]
        for sent, res in zip(batch, label_batch(batch)):
            if res:
                rows.append({'text': sent, **res})
            else:
                n_fail += 1
        if len(rows) > 0 and len(rows) % 500 < batch_size:
            pd.DataFrame(rows).to_csv(CKPT_PATH, index=False)
    print(f'Failures: {n_fail}/{len(sentences)}')
    return pd.DataFrame(rows)


print('Inference ready')

## 5. Load GoEmotions Part 2 (rows 22000+)

In [ ]:
go = load_dataset('go_emotions', 'simplified', split='train')
all_texts = [row['text'] for row in go]
sentences = all_texts[PART_START:PART_END]
print(f'Sentences to label: {len(sentences):,} (index {PART_START}–end)')

## 6. Label GoEmotions Part 2

In [ ]:
if os.path.exists(CKPT_PATH):
    ckpt_df = pd.read_csv(CKPT_PATH)
    done = set(ckpt_df['text'].tolist())
    todo = [s for s in sentences if s not in done]
    print(f'Resuming: {len(ckpt_df):,} done, {len(todo):,} remaining')
else:
    ckpt_df, todo = pd.DataFrame(), sentences

new_df = label_dataset(todo)
labeled_p2 = pd.concat([ckpt_df, new_df], ignore_index=True)
clean_p2 = labeled_p2.dropna(subset=EMOTIONS)
final_p2 = clean_p2[clean_p2['confidence'] >= 0.2].copy().reset_index(drop=True)
final_p2.to_csv(SAVE_PATH_P2, index=False)
print(f'Part 2 saved: {len(final_p2):,} rows → {SAVE_PATH_P2}')

## 7. Load SemEval EI-reg Continuous Labels
SemEval 2018 Task 1 EI-reg provides continuous emotion intensity (0.0–1.0) for joy, sadness, fear, anger. No LLM needed — just convert to Plutchik format.

In [ ]:
def load_semeval_ei_reg():
    """
    Load SemEval 2018 Task 1 EI-reg for joy/sadness/fear/anger.
    Converts to 8D Plutchik format (other axes set to 0).
    """
    # EI-reg emotion → Plutchik axis (direct mapping)
    SEMEVAL_TO_PLUTCHIK = {
        'joy': 'joy',
        'sadness': 'sadness',
        'fear': 'fear',
        'anger': 'anger',
    }
    all_rows = []

    for semeval_emo, plutchik_emo in SEMEVAL_TO_PLUTCHIK.items():
        config = f'subtask1.english.{semeval_emo}'
        try:
            ds = load_dataset('sem_eval_2018_task_1', config, trust_remote_code=True)
        except Exception as e:
            print(f'Could not load SemEval {semeval_emo}: {e}')
            continue

        for split in ['train', 'validation', 'test']:
            if split not in ds:
                continue
            for row in ds[split]:
                text = row.get('Tweet', row.get('text', ''))
                intensity = float(row.get('Intensity Score', row.get('intensity', 0.0)))
                if not text:
                    continue
                # Build 8D vector: target emotion gets intensity, rest get 0
                record = {'text': text, 'confidence': intensity}
                for e in EMOTIONS:
                    record[e] = intensity if e == plutchik_emo else 0.0
                all_rows.append(record)

        print(f'  {semeval_emo}: loaded {sum(1 for r in all_rows if r[plutchik_emo] > 0)} rows')

    return pd.DataFrame(all_rows)


print('Loading SemEval EI-reg...')
semeval_df = load_semeval_ei_reg()

if len(semeval_df) == 0:
    print('SemEval load failed — generating soft synthetic labels from GoEmotions instead')
    # Fallback: use already-labeled GoEmotions with high confidence as proxy
    semeval_df = final_p2[final_p2['confidence'] > 0.6].sample(min(3000, len(final_p2))).copy()

semeval_df.to_csv(SAVE_PATH_SE, index=False)
print(f'SemEval saved: {len(semeval_df):,} rows → {SAVE_PATH_SE}')
semeval_df.head(3)

## 8. Summary

In [ ]:
print('=== NB2 COMPLETE ===')
print(f'GoEmotions part 2 : {len(final_p2):,} sentences')
print(f'SemEval EI-reg    : {len(semeval_df):,} sentences')
print()
print('Files written:')
for p in [SAVE_PATH_P2, SAVE_PATH_SE]:
    sz = os.path.getsize(p) / 1e6
    print(f'  {p}  ({sz:.1f} MB)')
print()
print('Share this notebook output as Kaggle dataset: plutchik-labels-p2')
print('Person 3 adds BOTH plutchik-labels-p1 and plutchik-labels-p2 as inputs to NB3.')